# V2-2 — thử giới hạn prompt và chống lặp trên validation
Dùng adapter đã train, đúng hai ca validation đầu tiên của pilot retry trước. Kết quả chỉ kiểm tra định dạng và vòng lặp, chưa đánh giá độ đúng y khoa.

In [ ]:
from pathlib import Path
import subprocess, sys, json, hashlib
CODE_SHA = "a17c7cf"
REPO = Path("/kaggle/working/repo")
subprocess.run(["git", "clone", "https://github.com/kttt294/MRI-report-generator.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", CODE_SHA], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-kaggle.txt")], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), "Bật GPU T4 trong Kaggle Settings"
protocol = json.loads((REPO / "configs/v2_2_test_protocol.json").read_text(encoding="utf-8"))
adapters = [p.parent for p in Path("/kaggle/input").rglob("adapter_model.safetensors") if hashlib.sha256(p.read_bytes()).hexdigest() == protocol["adapter_sha256"]]
assert len(adapters) == 1, f"Cần đúng một adapter V2-2; tìm được {len(adapters)}"
RUN_DIR = Path("/kaggle/working/runs/v2-2-bounded-val-01")
subprocess.run([sys.executable, "scripts/v2_2_bounded_decode_validation.py", "--annotations-root", "/kaggle/input", "--adapter-root", str(adapters[0]), "--protocol", "configs/v2_2_test_protocol.json", "--output", str(RUN_DIR), "--limit", "2"], cwd=REPO, check=True)
print((RUN_DIR / "summary.json").read_text(encoding="utf-8"))
